In [1]:
import requests
import json
import os
import time

# Configuración inicial
url = "https://opencritic-api.p.rapidapi.com/review/game/15374"
headers = {
    "x-rapidapi-key": "e9c179ee23msh72da72619c88c3ap1b5c15jsna9d65f393a27",
    "x-rapidapi-host": "opencritic-api.p.rapidapi.com",
    "Content-Type": "application/json"
}

todas_las_resenas = []
skip_actual = 0
limite_deseado = 100

print(f"Iniciando extracción...")

while len(todas_las_resenas) < limite_deseado:
    querystring = {"skip": str(skip_actual), "sort": "newest"}
    try:
        response = requests.get(url, headers=headers, params=querystring)
        if response.status_code == 200:
            datos = response.json()
            lista_resenas = datos if isinstance(datos, list) else datos.get('reviews', [])

            if not lista_resenas: break

            for item in lista_resenas:
                # --- LIMPIEZA DE DATOS ---
                # Extraemos y normalizamos valores
                medio = "Desconocido"
                outlet_data = item.get('Outlet') or item.get('outlet')
                if isinstance(outlet_data, dict):
                    medio = outlet_data.get('name', 'Desconocido')

                # Solo guardamos si tenemos información relevante
                comentario = item.get('snippet', '').strip()
                puntuacion = item.get('score')

                if comentario or puntuacion:
                    todas_las_resenas.append({
                        'medio': medio,
                        'puntuacion': puntuacion if puntuacion is not None else "N/A",
                        'comentario': comentario.replace('\n', ' ')
                    })

                if len(todas_las_resenas) >= limite_deseado: break

            skip_actual += 20
            time.sleep(1)
        else:
            break
    except Exception as e:
        print(f"Error: {e}")
        break

# =================================================================
# GUARDADO EN JSON LIMPIO
# =================================================================
if todas_las_resenas:
    ruta_archivo = os.path.join(os.getcwd(), "resenas_limpias.json")

    with open(ruta_archivo, mode='w', encoding='utf-8') as file:
        # indent=4 genera un JSON "bonito" y fácil de leer
        json.dump(todas_las_resenas, file, ensure_ascii=False, indent=4)

    print(f"¡Éxito! Archivo guardado con {len(todas_las_resenas)} registros en: {ruta_archivo}")

Iniciando extracción...
¡Éxito! Archivo guardado con 63 registros en: c:\Users\jimmy\Downloads\resenas_limpias.json


In [2]:
import requests
import pandas as pd
import os
import time

# Configuración inicial
url = "https://opencritic-api.p.rapidapi.com/review/game/15374"
headers = {
    "x-rapidapi-key": "e9c179ee23msh72da72619c88c3ap1b5c15jsna9d65f393a27",
    "x-rapidapi-host": "opencritic-api.p.rapidapi.com",
    "Content-Type": "application/json"
}

todas_las_resenas = []
skip_actual = 0
limite_deseado = 100

print(f"Iniciando extracción y limpieza...")

while len(todas_las_resenas) < limite_deseado:
    querystring = {"skip": str(skip_actual), "sort": "newest"}
    try:
        response = requests.get(url, headers=headers, params=querystring)
        if response.status_code == 200:
            datos = response.json()
            lista_resenas = datos if isinstance(datos, list) else datos.get('reviews', [])
            if not lista_resenas: break

            for item in lista_resenas:
                medio = "Desconocido"
                outlet_data = item.get('Outlet') or item.get('outlet')
                if isinstance(outlet_data, dict):
                    medio = outlet_data.get('name', 'Desconocido')

                puntuacion = item.get('score')
                comentario = item.get('snippet', '').strip().replace('\n', ' ')

                # Solo agregamos si hay algo de información útil
                if comentario or puntuacion is not None:
                    todas_las_resenas.append({
                        'medio': medio,
                        'puntuacion': float(puntuacion) if puntuacion is not None else 0.0,
                        'comentario': comentario if comentario != "" else "Sin comentario"
                    })

                if len(todas_las_resenas) >= limite_deseado: break
            skip_actual += 20
            time.sleep(1)
        else: break
    except Exception as e:
        print(f"Error: {e}")
        break

# =================================================================
# PROCESAMIENTO, LIMPIEZA Y ORDENAMIENTO
# =================================================================
if todas_las_resenas:
    df = pd.DataFrame(todas_las_resenas)

    # 1. Limpieza: Eliminar duplicados exactos
    df = df.drop_duplicates()

    # 2. Limpieza: Asegurar tipos de datos
    df['puntuacion'] = pd.to_numeric(df['puntuacion'], errors='coerce').fillna(0.0)

    # 3. Ordenamiento:
    # Añadimos longitud de comentario para el criterio de orden
    df['longitud_comentario'] = df['comentario'].apply(len)
    df = df.sort_values(by=['puntuacion', 'longitud_comentario'], ascending=[False, False])

    # 4. Limpieza final: Eliminar la columna auxiliar
    df_final = df.drop(columns=['longitud_comentario'])

    # Guardar en CSV con formato limpio
    ruta_csv = os.path.join(os.getcwd(), "resenas_finales_limpias.csv")
    df_final.to_csv(ruta_csv, index=False, encoding='utf-8-sig')

    print(f"¡Éxito! Archivo guardado con {len(df_final)} registros únicos en: {ruta_csv}")
else:
    print("No se recolectaron datos.")

Iniciando extracción y limpieza...
¡Éxito! Archivo guardado con 63 registros únicos en: c:\Users\jimmy\Downloads\resenas_finales_limpias.csv


In [3]:
import requests
import pandas as pd
import os
import time

# --- FUNCIÓN DE EVALUACIÓN DE IA ---
def calcular_indice_ia(texto):
    score = 0
    texto_lower = str(texto).lower()

    # 1. Indicadores formales (conectores típicos de modelos de lenguaje)
    conectores = ["por otro lado", "en conclusión", "es importante destacar", "sin embargo", "en general", "cabe mencionar"]
    if any(conector in texto_lower for conector in conectores): score += 3

    # 2. Estructura sintáctica (IA tiende a oraciones largas y equilibradas)
    if len(texto_lower) > 80 and "." in texto_lower and texto_lower.count('.') >= 2: score += 3

    # 3. Falta de expresividad humana (ausencia de exclamaciones o dudas)
    if "!" not in texto_lower and "?" not in texto_lower: score += 2

    # 4. Longitud predecible
    if 60 < len(texto_lower) < 250: score += 2

    return min(score, 10)

# --- EXTRACCIÓN Y LIMPIEZA ---
# (Mantenemos tu lógica de extracción de 100 reseñas)
# ... [El código de requests va aquí como en los ejemplos anteriores] ...

# --- GENERACIÓN DE LA TABLA DE MÉTRICAS ---
if todas_las_resenas:
    df = pd.DataFrame(todas_las_resenas)

    # Limpieza inicial
    df = df.drop_duplicates()
    df['puntuacion'] = pd.to_numeric(df['puntuacion'], errors='coerce').fillna(0.0)

    # Aplicar la métrica de IA como una nueva columna
    df['indice_ia'] = df['comentario'].apply(calcular_indice_ia)

    # Ordenar según los criterios solicitados (Puntuación alta -> Índice IA)
    df_metricas = df.sort_values(by=['puntuacion', 'indice_ia'], ascending=[False, True])

    # Guardar la nueva tabla de métricas
    ruta_final = os.path.join(os.getcwd(), "tabla_metricas_resenas.csv")
    df_metricas.to_csv(ruta_final, index=False, encoding='utf-8-sig')

    print(f"¡Tabla de métricas creada con éxito!")
    print(f"Archivo guardado en: {ruta_final}")

    # Mostrar vista previa
    print("\n--- Vista previa de la Tabla de Métricas ---")
    print(df_metricas[['medio', 'puntuacion', 'indice_ia']].head(10))
else:
    print("No se encontraron reseñas para generar la tabla.")

¡Tabla de métricas creada con éxito!
Archivo guardado en: c:\Users\jimmy\Downloads\tabla_metricas_resenas.csv

--- Vista previa de la Tabla de Métricas ---
             medio  puntuacion  indice_ia
38      Gaming Age       100.0          3
61      Bastidores       100.0          4
23  Saving Content       100.0          5
62       Portal E7       100.0          5
39  Digital Chumps        98.0          7
21  Player2.net.au        91.0          5
36    Enternity.gr        90.0          4
0    Eternal Games        90.0          5
15  Rectify Gaming        90.0          5
26        NoobFeed        90.0          5


In [6]:
%pip install pandas deep-translator

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
import os
import time
from deep_translator import GoogleTranslator

def traducir_csv_resenas(nombre_archivo, columna_resena='review', idioma_destino='en'):
    """
    Lee un archivo CSV, traduce una columna específica y guarda el resultado en un nuevo archivo.
    """
    ruta_entrada = os.path.join(os.getcwd(), nombre_archivo)
    
    # Verificar si el archivo existe
    if not os.path.exists(ruta_entrada):
        print(f"❌ Error: No se encontró el archivo '{nombre_archivo}' en {os.getcwd()}")
        return

    try:
        # Cargar el CSV
        print(f"📂 Cargando archivo: {nombre_archivo}...")
        df = pd.read_csv(ruta_entrada)
        
        # Verificar que la columna exista
        if columna_resena not in df.columns:
            print(f"❌ Error: La columna '{columna_resena}' no existe en el archivo.")
            print(f"Columnas disponibles: {list(df.columns)}")
            return

        print(f"🔄 Iniciando traducción de {len(df)} reseñas al idioma '{idioma_destino}'...")
        traductor = GoogleTranslator(source='auto', target=idioma_destino)
        traducciones = []

        # Traducir fila por fila con manejo de errores
        for index, texto in enumerate(df[columna_resena]):
            try:
                if pd.isna(texto) or str(texto).strip() == "":
                    traducciones.append("")
                else:
                    traduccion = traductor.translate(str(texto))
                    traducciones.append(traduccion)
                
                # Pequeño retraso para evitar bloqueos por límite de peticiones (Rate Limit)
                time.sleep(0.5) 
                
                # Mostrar progreso cada 10 reseñas
                if (index + 1) % 10 == 0:
                    print(f"   ⏳ Progreso: {index + 1}/{len(df)} reseñas traducidas.")

            except Exception as e:
                print(f"⚠️ Error al traducir la fila {index}: {e}")
                traducciones.append(texto) # Mantiene el original si falla

        # Añadir la nueva columna al DataFrame
        nueva_columna = f"{columna_resena}_traducida"
        df[nueva_columna] = traducciones

        # Guardar en un nuevo archivo CSV
        nombre_salida = nombre_archivo.replace('.csv', '_traducido.csv')
        ruta_salida = os.path.join(os.getcwd(), nombre_salida)
        df.to_csv(ruta_salida, index=False, encoding='utf-8-sig')

        print(f"✅ Proceso completado exitosamente.")
        print(f"💾 Archivo guardado en: {ruta_salida}")

    except Exception as e:
        print(f"❌ Error general en el procesamiento: {e}")

if __name__ == "__main__":
    # Asegúrate de colocar el nombre exacto de tu archivo y la columna que contiene el texto.
    # Por ejemplo, para el archivo de Steam generado anteriormente:
    traducir_csv_resenas(
        nombre_archivo="tabla_metricas_resenas.csv", 
        columna_resena="comentario",      # Cambia esto si en el otro archivo la columna se llama distinto (ej. 'Resena')
        idioma_destino="es"           # 'en' para inglés, 'es' para español, etc.
    )

📂 Cargando archivo: tabla_metricas_resenas.csv...
🔄 Iniciando traducción de 63 reseñas al idioma 'es'...
   ⏳ Progreso: 10/63 reseñas traducidas.
   ⏳ Progreso: 20/63 reseñas traducidas.
   ⏳ Progreso: 30/63 reseñas traducidas.
   ⏳ Progreso: 40/63 reseñas traducidas.
   ⏳ Progreso: 50/63 reseñas traducidas.
   ⏳ Progreso: 60/63 reseñas traducidas.
✅ Proceso completado exitosamente.
💾 Archivo guardado en: c:\Users\jimmy\Downloads\tabla_metricas_resenas_traducido.csv
